# LMTA - Export ALMP XML

This notebook orchestrates one ALMP XML export:

1. Reads one frozen `submission_id` from the seven `almp` tables in the LMTA Warehouse using PySpark.
2. Converts only that filtered submission to ordinary Python dictionaries.
3. Calls `almp_xml_export.py` to build, validate, and save the XML in ICT_DL1.

**Before running:** attach `ICT_DL1` as the notebook's default Lakehouse and upload the Python module and XSD files to the documented folders.


In [ ]:
# Run once per new notebook session. If lxml is already available, this is quick.
%pip install lxml


After `%pip install`, restart the notebook session only if Fabric requests it, then continue below.


In [ ]:
from pathlib import Path
import importlib.util
import json

MODULE_PATH = Path("/lakehouse/default/Files/LMTA/code/python/almp_xml_export.py")
if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Upload the Python module to: {MODULE_PATH}")

spec = importlib.util.spec_from_file_location("almp_xml_export", MODULE_PATH)
almp_xml_export = importlib.util.module_from_spec(spec)
spec.loader.exec_module(almp_xml_export)

ExportConfig = almp_xml_export.ExportConfig
export_submission = almp_xml_export.export_submission
print(f"Loaded: {MODULE_PATH}")


In [ ]:
# EDIT THESE VALUES
SUBMISSION_ID = 101
WAREHOUSE_NAME = "LMTA"
PROGRAM_TYPE = "LMDA"     # LMDA or WDA
SEQUENCE_NUMBER = 1
MAIN_XSD_NAME = "ALMP.xsd"

# If your Fabric environment requires a workspace-qualified identifier, use:
# WAREHOUSE_NAME = "<WorkspaceName>.LMTA"


In [ ]:
from pyspark.sql import functions as F

TABLES = {
    "submission": "almp.submission",
    "sys_action": "almp.sys_action",
    "client": "almp.client",
    "action_plan": "almp.action_plan",
    "followup": "almp.action_plan_followup",
    "intervention": "almp.intervention",
    "support": "almp.intervention_support",
}

SORT_COLUMNS = (
    "sort_seq", "client_key", "action_plan_key", "followup_key",
    "intervention_key", "support_key", "sys_action_key",
)


def read_submission_dataset(logical_name, object_name, submission_id):
    identifier = f"{WAREHOUSE_NAME}.{object_name}"
    df = spark.read.synapsesql(identifier)

    if "submission_id" in df.columns:
        id_column = "submission_id"
    elif "batch_id" in df.columns:
        id_column = "batch_id"
    else:
        raise ValueError(f"{identifier} has neither submission_id nor batch_id")

    filtered = df.filter(F.col(id_column) == F.lit(submission_id))
    ordering = [c for c in SORT_COLUMNS if c in filtered.columns]
    if ordering:
        filtered = filtered.orderBy(*ordering)

    rows = [row.asDict(recursive=True) for row in filtered.collect()]
    print(f"{logical_name}: {len(rows):,} row(s)")
    return rows


data = {
    logical_name: read_submission_dataset(logical_name, object_name, SUBMISSION_ID)
    for logical_name, object_name in TABLES.items()
}

if len(data["submission"]) != 1:
    raise ValueError(
        f"Expected one almp.submission row for submission_id={SUBMISSION_ID}; "
        f"found {len(data['submission'])}."
    )


In [ ]:
# IMPORTANT: For production, populate exact element order from the XSD xs:sequence.
# The examples below are placeholders and are intentionally commented out.
ELEMENT_ORDER = {
    # "submission": ["SchemaVersion", "agreementHolderName", "sysComment"],
    # "client": ["clientID", "socialInsuranceNumber", "lastName", "firstName"],
    # "action_plan": [...],
    # "followup": [...],
    # "intervention": [...],
    # "support": [...],
    # "sys_action": [...],
}

config = ExportConfig(
    submission_id=SUBMISSION_ID,
    program_type=PROGRAM_TYPE,
    sequence_number=SEQUENCE_NUMBER,
    lakehouse_root="/lakehouse/default/Files/LMTA",
    main_xsd_name=MAIN_XSD_NAME,
    fail_on_validation_error=True,
    table_element_order=ELEMENT_ORDER,
)

print("XSD:", config.xsd_dir / config.main_xsd_name)
print("Submitted output:", config.submitted_dir)
print("Rejected output:", config.rejected_dir)


In [ ]:
# Generate, validate, and file the XML.
try:
    result = export_submission(data, config)
    print(json.dumps(result, indent=2, default=str))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
    raise


## Expected output locations

- Valid XML: `Files/LMTA/output/xml/submitted/`
- Invalid XML: `Files/LMTA/output/xml/rejected/`
- Validation report: `Files/LMTA/output/validation/`

The exporter never overwrites an existing output file.
